# Medical knowledge chunking playground    Use this notebook to experiment with cleaning, sectioning, and chunking medical guideline PDFs (or text exports) before feeding them into the RAG pipeline. Update the paths and chunking parameters to mirror your environment, then inspect how the resulting chunks look before exporting to JSONL.

## Prerequisites    - Install either `pypdf` or `PyPDF2` so `_extract_pdf_text` can read PDFs.    - Point `knowledge_dir` to your `Medical_Knowledge` folder (or a directory of `.txt` exports).    - Optional: install `matplotlib` if you want to visualize chunk length distributions.

In [ ]:
from pathlib import Pathimport jsonimport statisticsimport textwrapfrom evidence_rl.ingestion import (    _extract_pdf_text,    _iter_sections,    chunk_guideline_text,    export_documents_jsonl,)from evidence_rl.documents import Document

In [ ]:
# Path to your clinical guideline PDFs or text exportsknowledge_dir = Path("/path/to/Medical_Knowledge")# Chunking configuration you want to experiment withchunk_size = 400  # words per chunkoverlap = 80     # word overlap between chunks

In [ ]:
if not knowledge_dir.exists():    raise FileNotFoundError(f"Knowledge directory not found: {knowledge_dir}")files = sorted([p for p in knowledge_dir.glob('**/*') if p.suffix.lower() in {'.pdf', '.txt'}])print(f'Found {len(files)} files under {knowledge_dir}')for path in files[:5]:    print(f' - {path}')

### Peek at raw text from a sample file    Change `sample_path` if you want to inspect a different file. Trimming keeps the output manageable.

In [ ]:
sample_path = files[0]raw_text = _extract_pdf_text(sample_path)print(f'Sampled file: {sample_path.name}')print(textwrap.shorten(raw_text.replace('', ' '), width=2000, placeholder='...'))

### Inspect detected sections    Section detection uses numbered headings or uppercase titles as heuristics.

In [ ]:
sections = list(_iter_sections(raw_text))print(f'Detected {len(sections)} sections')for title, body in sections[:5]:    preview = textwrap.shorten(body.replace('', ' '), width=240, placeholder='...')    print(f"[{title}]{preview}")

### Chunk the guideline text    Adjust `chunk_size` and `overlap` above to see how the chunking changes.

In [ ]:
chunks = chunk_guideline_text(    raw_text,    source_id=sample_path.stem,    chunk_size=chunk_size,    overlap=overlap,)print(f'Generated {len(chunks)} chunks from {sample_path.name}')print(f'First chunk metadata: {chunks[0].metadata}')print(f"First chunk preview: {textwrap.shorten(chunks[0].text, width=280, placeholder='...')}")

### Chunk length statistics    Quick summary of tokenized word counts per chunk.

In [ ]:
word_counts = [len(chunk.text.split()) for chunk in chunks]print({    'min_words': min(word_counts),    'max_words': max(word_counts),    'mean_words': round(statistics.mean(word_counts), 2),    'median_words': statistics.median(word_counts),})

Optional: visualize the distribution. This cell will no-op if `matplotlib` is missing.

In [ ]:
try:    import matplotlib.pyplot as pltexcept ImportError:    print('matplotlib not installed; skipping histogram')else:    plt.hist(word_counts, bins=20, color='steelblue', edgecolor='black')    plt.title('Chunk word counts')    plt.xlabel('Words')    plt.ylabel('Frequency')    plt.show()

### Inspect a few chunks    Helpful when tuning overlaps to ensure sections stay coherent.

In [ ]:
for chunk in chunks[:3]:    print(f"ID: {chunk.doc_id}")    print(f"Section: {chunk.metadata.get('section_title')} (index {chunk.metadata.get('section_index')})")    print(textwrap.fill(chunk.text, width=100))    print('-' * 80)

### Export the chunked corpus to JSONL    Uncomment the cell below to write a JSONL file that the retrieval pipeline can ingest.

In [ ]:
# output_path = knowledge_dir / 'chunked_guidelines.jsonl'# documents = [chunk.to_document() for chunk in chunks]# export_documents_jsonl(documents, output_path)# print(f'Wrote {len(documents)} chunks to {output_path}')

### Reload and sanity check the JSONL output    This can help confirm the exported structure matches what the retriever expects.

In [ ]:
# if 'output_path' in globals():#     reloaded: list[Document] = []#     with open(output_path, 'r', encoding='utf-8') as handle:#         for line in handle:#             payload = json.loads(line)#             reloaded.append(Document(**payload))#     print(f'Reloaded {len(reloaded)} documents; first entry:')#     print(reloaded[0])